# 10 — Runtime topology design

**Goal:** design a fresh topology **per request**, not just offline.

Notebook 08 evolved topologies offline. You ran the search once, picked a winner, and shipped it. That's great when you know in advance what kinds of requests will come in.

But many real systems get a wide variety of requests. A summarize question, a verification question, a calculation, a search. You don't want **one** fixed topology — you want the system to pick a fitting topology **per request**.

That is what this notebook builds:

1. A **designer agent** that picks a topology for each incoming goal.
2. A **`TopologyOrchestrator`** that hydrates and runs whatever the designer produced.
3. A **cache** so similar goals reuse a previously-designed topology (saves an LLM call).
4. A **seed library** — a small list of known-good topologies passed to the designer as hints.
5. **Online learning** — failed runs decay the cache, so bad topologies stop being reused.

This closes the loop between offline search (notebooks 08–09) and runtime spawning (notebook 02).

**Cost: ~$1–2.**


## 1. Setup — agents + designer

Three agents, all simple. The same trick as before: same class, different prompts.


In [1]:
from __future__ import annotations
from typing import Any
from pydantic import BaseModel, Field
from orqest import load_config
from orqest.agents import BaseAgent, GlobalState

config = load_config()
print(f"Task model: {config.llm_model}")


class Answer(BaseModel):
    answer: str = Field(description="Answer to the question.")


class CotAgent(BaseAgent[GlobalState, Answer]):
    async def _run_implementation(self, state, **kw):
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(latest, state)
        return result.output


def _make_factory(name: str, prompt: str):
    def _f():
        return CotAgent(
            agent_name=name,
            system_prompt=prompt,
            output_type=Answer,
            model=config.llm_model,
            api_key=config.llm_api_key,
        )
    return _f


agent_registry = {
    "cot":        _make_factory("cot",        "You answer questions concisely. If unsure, say so."),
    "verifier":   _make_factory("verifier",   "Re-check the candidate answer for correctness given the paragraph; correct if wrong; abstain if unanswerable."),
    "summarizer": _make_factory("summarizer", "Summarize the input in one sentence."),
}
print("agents:", sorted(agent_registry))

Task model: openai:gpt-5.2
agents: ['cot', 'summarizer', 'verifier']


### 1b. The designer agent

The designer is a `BaseAgent` whose output type is `TopologyDesign`. The output includes a typed `TopologySpec` plus a one-sentence `thought` explaining the choice.

Pydantic-AI handles the JSON schema for `TopologySpec` automatically — the LLM sees the full structure it needs to produce, so it can emit valid pipelines / parallels / routers as structured output.

The system prompt has one critical rule: **only use the allowed agent and callable names**. Hydration fails if the designer references an unknown name, and that failure becomes a signal to try again.


In [2]:
from orqest.optimization import TopologyDesign


class DesignerAgent(BaseAgent[GlobalState, TopologyDesign]):
    async def _run_implementation(self, state, **kw):
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(latest, state)
        return result.output


designer_agent = DesignerAgent(
    agent_name="designer",
    system_prompt=(
        "You design typed Orqest topologies (Pipeline / Parallel / Router / RefinementLoop) "
        "for incoming goals. Always emit a `thought` and a `spec`. Never emit raw Python. "
        "Use ONLY agent_name and callable_name values from the user message's allowlist; "
        "hydration will fail and your design will be discarded if you reference unknown names."
    ),
    output_type=TopologyDesign,
    model=config.llm_model,
    api_key=config.llm_api_key,
)
print("designer ready")

designer ready


## 2. Cold design — no cache

The simplest case: one LLM call, one topology, no caching. The `NoCache` cache is a no-op.

We give the designer a goal that mentions both "summarize" and "verify". A good designer should pick a two-step pipeline.


In [3]:
import time
from orqest.autonomy.runtime import NoCache, RuntimeTopologyDesigner
from orqest.orchestration.hydrate import CallableRegistry

callable_registry = CallableRegistry()        # empty — designer can only use agents


cold_designer = RuntimeTopologyDesigner(
    designer_agent=designer_agent,
    callable_registry=callable_registry,
    agent_registry=agent_registry,
    constraints="Prefer the simplest topology that fits. Add a verifier step only when the goal explicitly mentions 'verify' or 'check'.",
    cache=NoCache(),
)


t0 = time.monotonic()
goal = "Summarize the following paragraph and verify your summary stays faithful: 'Python was first released in 1991 by Guido van Rossum.'"
spec = await cold_designer.design(goal)
design_ms = (time.monotonic() - t0) * 1000

print(f"design_ms: {design_ms:.0f}")
print(f"spec.kind: {spec.kind}")
print(f"spec JSON:")
print(spec.model_dump_json(indent=2)[:600])

design_ms: 4310
spec.kind: pipeline
spec JSON:
{
  "kind": "pipeline",
  "steps": [
    {
      "operation": {
        "kind": "agent_step",
        "agent_name": "summarizer",
        "inline_spec": null
      },
      "config": null
    },
    {
      "operation": {
        "kind": "agent_step",
        "agent_name": "verifier",
        "inline_spec": null
      },
      "config": null
    }
  ],
  "name": "summarize_then_verify"
}


## 3. Caching — similar goals skip the design call

A real system gets many similar requests. Designing from scratch each time is wasteful.

`MemoryStoreCache` solves this. Every successful design is stored as a **semantic memory** in a `topology_cache` namespace. When a new goal arrives:

- Embed it.
- Look up the most similar prior goal.
- If similar enough (above `threshold`), reuse its topology.
- Otherwise, design fresh.

For the demo we use a simple fake embedder (character hashing) because we don't want to depend on a sentence-transformer here. In production you would plug in a real embedder.

Watch the timings: the **first** call pays for design (~3 seconds for an LLM call). The next two calls find a cached topology and return in milliseconds.


In [4]:
from orqest.memory import LocalMemoryStore
from orqest.autonomy.runtime import MemoryStoreCache


async def fake_embedder(text: str) -> list[float]:
    """Tiny demo embedder: maps to a 64-dim vector by char hashing.

    Real production code would use a sentence-transformer.
    """
    import hashlib
    h = hashlib.sha256(text.lower().encode()).digest()[:64]
    return [b / 255.0 for b in h]


store = LocalMemoryStore(":memory:", embedder=fake_embedder)

# threshold=0.0 means "always treat the top hit as a match" — for the demo only.
cache = MemoryStoreCache(store, threshold=0.0, namespace="topology_cache")


cached_designer = RuntimeTopologyDesigner(
    designer_agent=designer_agent,
    callable_registry=callable_registry,
    agent_registry=agent_registry,
    constraints="Prefer the simplest topology that fits.",
    cache=cache,
)


goals = [
    "Summarize the input paragraph.",
    "Briefly summarize this paragraph.",
    "Give me a one-sentence summary of this text.",
]

for g in goals:
    t0 = time.monotonic()
    spec = await cached_designer.design(g)
    elapsed = (time.monotonic() - t0) * 1000
    print(f"  goal: {g!r:<60} → kind={spec.kind:<10} {elapsed:>6.0f}ms")

stats = cached_designer.stats
print()
print(f"stats: lookups={stats.lookups} hits={stats.hits} misses={stats.misses} designs={stats.designs}")

  goal: 'Summarize the input paragraph.'                             → kind=pipeline     3550ms
  goal: 'Briefly summarize this paragraph.'                          → kind=pipeline        4ms
  goal: 'Give me a one-sentence summary of this text.'               → kind=pipeline        3ms

stats: lookups=3 hits=2 misses=1 designs=1


## 4. `TopologyOrchestrator` — design + execute in one call

The pieces above (designer + cache) give you a topology. But you still have to hydrate and run it.

`TopologyOrchestrator` wraps the whole flow:

1. Ask the designer for a topology (cached or fresh).
2. Hydrate it against the agent and callable registries.
3. Execute it on the goal.
4. Record success or failure to the cache.

It returns a `TopologyExecutionResult` with structural metrics (kind, depth), timing (design_ms, execution_ms), and a cache-hit signal.


In [5]:
from orqest.autonomy.topology_orchestrator import TopologyOrchestrator


orchestrator = TopologyOrchestrator(
    cached_designer,
    callable_registry=callable_registry,
    agent_registry=agent_registry,
)


goals_to_run = [
    "Summarize this paragraph: 'Python was first released in 1991.'",
    "What's a brief summary of: 'JavaScript first appeared in 1995.'",
    "Verify and answer concisely: 'Tesla, founded in 2003, has Elon Musk as CEO.' Question: who is the CEO?",
]

results = []
for g in goals_to_run:
    r = await orchestrator.execute(g)
    results.append(r)


print(f"{'goal':<60} {'kind':<10} {'cache':<6} {'design_ms':>9} {'exec_ms':>9}")
print("-" * 110)
for r in results:
    g_short = r.goal[:55] + "..." if len(r.goal) > 58 else r.goal
    print(f"{g_short:<60} {r.spec_kind:<10} {str(r.cache_hit):<6} {r.design_ms:>9.0f} {r.execution_ms:>9.0f}")

goal                                                         kind       cache  design_ms   exec_ms
--------------------------------------------------------------------------------------------------------------
Summarize this paragraph: 'Python was first released in...   pipeline   True           4      1487
What's a brief summary of: 'JavaScript first appeared i...   pipeline   True           3      1326
Verify and answer concisely: 'Tesla, founded in 2003, h...   pipeline   True           3      1326


## 5. Seed library — biasing the designer

In production you would typically run `MetaAgentSearch` (notebook 08) **offline** against a representative gold set. The winners are known-good topologies. You then pass them as a **seed library** to the runtime designer.

The designer sees these in its prompt and prefers composing or specializing them over inventing fresh structures. This gives you the best of both worlds: offline search finds good shapes, runtime designer picks the right one per request.


In [6]:
from orqest.orchestration.spec import AgentStepSpec, PipelineSpec, PipelineStepEntry


seed_library = [
    # Known-good: a 2-step pipeline for "verify-style" goals.
    PipelineSpec(
        name="cot_then_verify",
        steps=[
            PipelineStepEntry(operation=AgentStepSpec(agent_name="cot")),
            PipelineStepEntry(operation=AgentStepSpec(agent_name="verifier")),
        ],
    ),
    # Known-good: a single summarizer for "summarize-style" goals.
    PipelineSpec(
        name="single_summarizer",
        steps=[PipelineStepEntry(operation=AgentStepSpec(agent_name="summarizer"))],
    ),
]


seeded_designer = RuntimeTopologyDesigner(
    designer_agent=designer_agent,
    callable_registry=callable_registry,
    agent_registry=agent_registry,
    constraints="Prefer the simplest topology. Use the library entries when they fit.",
    cache=NoCache(),                                # cache off so we see the seeded output
    seed_library=seed_library,
)


goal = "Summarize and then verify: 'Berlin, the capital of Germany, has 3.7 million people.'"
spec = await seeded_designer.design(goal)

print(f"designed kind: {spec.kind}")
print(f"spec JSON:")
print(spec.model_dump_json(indent=2)[:800])

designed kind: pipeline
spec JSON:
{
  "kind": "pipeline",
  "steps": [
    {
      "operation": {
        "kind": "agent_step",
        "agent_name": "summarizer",
        "inline_spec": null
      },
      "config": null
    },
    {
      "operation": {
        "kind": "agent_step",
        "agent_name": "verifier",
        "inline_spec": null
      },
      "config": null
    }
  ],
  "name": "summarize_then_verify"
}


## 6. Online learning — failures decay the cache

A cached topology is only useful if it **works**. If a hydrated topology crashes during execution, the orchestrator records the failure to the cache. The memory store decays that entry's **reliability score**.

With a strict `min_reliability` filter, future similar requests skip the decayed entry and re-design from scratch. That's the online learning loop: the system **learns which topologies to stop using**.

In this demo we seed the cache with a deliberately bad topology that references a nonexistent agent (`ghost`). The orchestrator tries to run it, hydration fails, the cache reliability decays, and a strict filter no longer returns it.


In [7]:
# Fresh cache + designer so the decay is easy to see.
decay_store = LocalMemoryStore(":memory:", embedder=fake_embedder)
decay_cache = MemoryStoreCache(decay_store, threshold=0.0, min_reliability=0.0, namespace="topology_cache")


# Seed the cache with a bad topology — references a non-existent agent.
bad_spec = PipelineSpec(steps=[PipelineStepEntry(operation=AgentStepSpec(agent_name="ghost"))])
await decay_cache.store("do something", bad_spec, success=True)
print("cached bad spec; lookup OK?", (await decay_cache.lookup("do something")) is not None)


decay_designer = RuntimeTopologyDesigner(
    designer_agent=designer_agent,
    callable_registry=callable_registry,
    agent_registry=agent_registry,
    cache=decay_cache,
    verify_on_hit=False,                 # disabled so the bad cache hit reaches execution
)
decay_orch = TopologyOrchestrator(
    decay_designer,
    callable_registry=callable_registry,
    agent_registry=agent_registry,
)


# Execute — will fail because 'ghost' is not in the registry.
try:
    await decay_orch.execute("do something")
except Exception as exc:
    print(f"execution failed (expected): {type(exc).__name__}")


# Now use a strict min_reliability filter — the decayed entry should NOT be returned.
strict_cache = MemoryStoreCache(decay_store, threshold=0.0, min_reliability=0.9, namespace="topology_cache")
recalled = await strict_cache.lookup("do something")
print(f"strict filter recall after decay: {recalled}")     # None

cached bad spec; lookup OK? True
execution failed (expected): KeyError
strict filter recall after decay: None


## Recap

The full pattern:

1. **Designer agent** — takes a goal, emits a `TopologySpec`. Output is typed JSON, not Python.
2. **`TopologyOrchestrator`** — designs, hydrates, runs, records.
3. **Cache** — similar goals reuse previously designed topologies. Saves an LLM call per repeat.
4. **Seed library** — pass offline-evolved topologies as hints. Designer prefers composing them.
5. **Reliability decay** — failed runs lower a cache entry's score. Strict filters then skip it.

**Why this matters.** It closes the loop between offline search and runtime spawning. Offline search finds good shapes. The runtime designer picks one per request. Caching makes it cheap. Decay makes it self-correcting.

**What's next:**

- **Sandboxed code generation** — for cases where typed specs are too constrained.
- **Retrieval policy** — fetch from a vector store of topologies instead of holding the whole library in the prompt.
- **Output-quality decay** — today decay only fires on execution exceptions. Wiring a quality judge would let consumers decay topologies that produce *bad outputs*, not just crashing ones.
- **Confidence integration** — feed `metacognition.confidence` into the reliability signal.

See the **Runtime topology design** section in `docs/concepts/topology_optimization.md` for the design rationale.
